# Static image rendering (Swing & JavaFX)

This exercises PR #123: `display(component)` / `display(node)` — and a bare last-expression value —
render a GUI component to a **still PNG**. No JupyterLab extension, no comms; the image shows in any
frontend and on GitHub/nbviewer.

- **Swing/AWT** is headless-native (no display needed).
- **JavaFX** is optional (pull it with `%maven`) and needs a display to snapshot — on a headless host run
  the kernel under **Xvfb** (`xvfb-run -a jupyter lab`). Build FX nodes on the JavaFX thread (helper below).

Rebuild the kernel from the `static-image-rendering` branch and reinstall the kernelspec before running.

## Swing / AWT

In [ ]:
import javax.swing.*;
import java.awt.*;

// A laid-out Swing component renders as a static image.
JButton button = new JButton("Hello from Swing");
button.setSize(220, 48);
display(button);

In [ ]:
import javax.swing.*;
import java.awt.*;

// Custom painting also works - here a tiny bar chart.
JPanel chart = new JPanel() {
    protected void paintComponent(Graphics g) {
        super.paintComponent(g);
        Graphics2D g2 = (Graphics2D) g;
        g2.setRenderingHint(RenderingHints.KEY_ANTIALIASING, RenderingHints.VALUE_ANTIALIAS_ON);
        int[] vals = {40, 90, 60, 120, 80};
        int w = getWidth() / vals.length;
        for (int i = 0; i < vals.length; i++) {
            g2.setColor(Color.getHSBColor(i / (float) vals.length, 0.6f, 0.9f));
            g2.fillRect(i * w + 8, getHeight() - vals[i] - 10, w - 16, vals[i]);
        }
        g2.setColor(Color.DARK_GRAY);
        g2.drawString("Swing bar chart", 8, 16);
    }
};
chart.setBackground(Color.WHITE);
chart.setSize(360, 180);
display(chart);

## JavaFX (optional)

Pull OpenJFX (edit the classifier for your OS: `linux` | `mac` | `mac-aarch64` | `win`). JavaFX needs a
display; on a headless host run the kernel under Xvfb.

All JavaFX imports + the `fx(...)` helper live in the setup cell. The cells below pass the node **straight to
`display(...)`** (no intermediate variable) — storing a JavaFX-typed top-level variable trips a JShell
field-resolution bug for runtime-loaded types.

In [ ]:
%maven org.openjfx:javafx-base:jar:linux:21.0.2
%maven org.openjfx:javafx-graphics:jar:linux:21.0.2
%maven org.openjfx:javafx-controls:jar:linux:21.0.2
%maven org.openjfx:javafx-swing:jar:linux:21.0.2

import javafx.application.Platform;
import javafx.embed.swing.JFXPanel;
import javafx.scene.layout.Pane;
import javafx.scene.shape.Circle;
import javafx.scene.shape.Rectangle;
import javafx.scene.paint.Color;
import javafx.scene.chart.BarChart;
import javafx.scene.chart.CategoryAxis;
import javafx.scene.chart.NumberAxis;
import javafx.scene.chart.XYChart;
import java.util.concurrent.CountDownLatch;
import java.util.function.Supplier;

new JFXPanel(); // boots the JavaFX toolkit (needs a display / Xvfb)

// Build a JavaFX node on the FX thread and return it (as Object - do not store it in a typed var).
Object fx(Supplier<Object> builder) throws Exception {
    Object[] holder = new Object[1];
    CountDownLatch latch = new CountDownLatch(1);
    Platform.runLater(() -> { try { holder[0] = builder.get(); } finally { latch.countDown(); } });
    latch.await();
    return holder[0];
}
System.out.println("JavaFX " + System.getProperty("javafx.runtime.version") + " ready");

In [ ]:
// Pass the node straight to display(...) - no intermediate variable.
display(fx(() -> {
    Pane p = new Pane();
    p.setPrefSize(300, 200);
    Rectangle r = new Rectangle(40, 50, 90, 90);
    r.setFill(Color.ORANGE);
    Circle c = new Circle(170, 100, 70, Color.CORNFLOWERBLUE);
    p.getChildren().addAll(r, c);
    return p;
}));

In [ ]:
display(fx(() -> {
    BarChart<String, Number> bc = new BarChart<>(new CategoryAxis(), new NumberAxis());
    bc.setTitle("JavaFX BarChart");
    XYChart.Series<String, Number> s = new XYChart.Series<>();
    s.setName("demo");
    s.getData().add(new XYChart.Data<>("A", 40));
    s.getData().add(new XYChart.Data<>("B", 90));
    s.getData().add(new XYChart.Data<>("C", 60));
    bc.getData().add(s);
    bc.setPrefSize(440, 300);
    return bc;
}));